# Leptonic signal/background classification with TMVA

This notebook trains a binary Boosted Decision Tree (BDT) to distinguish the leptonic \(W^+W^-\) signal from the electron/muon background. It also builds BDT-score templates from an independent holdout sample and performs a binned likelihood fit to those templates.

## Input data and requirements

The notebook uses the four leptonic decay samples and the leptonic background configured under `LEPTONIC_SAMPLE_FILES` in `analysis_config.py`. The input ROOT files must contain the LHE particle information listed in the repository README. The notebook first looks for the tree configured by `FCC_BDT_TREE` (default `TreeS`) and can fall back to a tree containing the `nLHEPart` branch.

ROOT must be available through PyROOT, and the ROOT installation must include TMVA. Install the additional Python dependencies from `requirements.txt` in the same environment used by Jupyter.

## Training and holdout strategy

The BDT input variables are leading-lepton transverse momentum (`lep_pt`), dilepton invariant mass (`mll`), and the observable decay angle (`cos_theta_star`). Each source file is split before event selection: 70% of events are assigned to the TMVA training/test sample and 30% are reserved for independent BDT-score templates. The file-specific split is reproducible for a fixed filename and `SPLIT_SEED`.

Within the 70% training sample, TMVA performs its own training/test division. The reserved 30% holdout sample is not used in that training or test division; it is used later to construct the likelihood templates.

The `SOLO_E_MU` setting in the setup cell controls the background flavour selection. Its default is `False`, retaining all supported two-charged-lepton electron/muon final states. Set it to `True` to select only the \(e^+\mu^-\) state.

## How to run and outputs

Configure the sample names and physics settings in `analysis_config.py`, then run all cells from top to bottom. The notebook writes intermediate ROOT files and `TMVA_Binario_Output.root` under `outputs/`. TMVA stores its weight file in `dataset/weights/`; test distributions, BDT templates, and likelihood results are displayed in the notebook, with plots also saved under `outputs/plots/`.


In [ ]:
# 1. Setup and configuration
import array
import math
import os
import random
import zlib
from bisect import bisect_right

import numpy as np
from iminuit import Minuit

# PyROOT must be available in the active Python environment.
import ROOT

from analysis_config import (
    BDT_TREE_NAME,
    COLLISION_ENERGY_GEV,
    BINS_BDT as CONFIG_BINS_BDT,
    CROSS_SECTIONS_PB,
    EVENTS_TREE_NAME,
    REBIN_BDT as CONFIG_REBIN_BDT,
    LEPTONIC_SAMPLE_FILES,
    LUMI_PB,
    PROJECT_DIR,
    output_path,
    resolve_input,
)

# Keep TMVA's relative dataset/weights directory inside the project.
os.chdir(PROJECT_DIR)

ROOT.TMVA.Tools.Instance()

# ---------------- CONFIGURAZIONE ----------------
TRAIN_FRACTION = 0.70
SPLIT_SEED = 20261026
BINS_BDT = CONFIG_BINS_BDT
REBIN_BDT = CONFIG_REBIN_BDT

SIGMA_LL = CROSS_SECTIONS_PB["00"]
SIGMA_TT = CROSS_SECTIONS_PB["TT"]
SIGMA_0T = CROSS_SECTIONS_PB["0T"]
SIGMA_T0 = CROSS_SECTIONS_PB["0T"]
SIGMA_BKG = CROSS_SECTIONS_PB["background"]

# Background selection:
# True  = only e+ mu-
# False = all final states with exactly two charged e/mu leptons
SOLO_E_MU = False

# Holdout fraction for each source file, assigned before event selection.
holdout_fraction_by_file = {}

# Keep ROOT graphics objects alive for the following cells.
ROOT_KEEPALIVE = []

In [ ]:
# Shared plot styling and ROOT graphics helpers.
# Text and line sizes are fractions of the pad height. Adjust them here
# to change the readability of the exported figures.
SIZE_LABEL  = 0.045     # numeri sugli assi
SIZE_TITLE  = 0.050     # titoli degli assi
SIZE_LEGEND = 0.036     # testo delle legende
SIZE_FCC    = 0.045     # scritta "FCC-ee Simulation" (l'energia e' un po' piu' piccola)
LINE_WIDTH  = 3         # spessore delle linee degli istogrammi

# --- Impostazioni globali di ROOT ---
ROOT.TH1.AddDirectory(False)        # gli istogrammi non dipendono dai file aperti
ROOT.gStyle.SetOptStat(0)           # niente riquadro delle statistiche
ROOT.gStyle.SetOptTitle(0)          # niente titolo in cima ai grafici
ROOT.gStyle.SetPadTickX(1)          # tacche anche sull'asse superiore
ROOT.gStyle.SetPadTickY(1)          # tacche anche sull'asse destro
ROOT.gStyle.SetEndErrorSize(0)      # barre d'errore senza cappuccio
ROOT.gStyle.SetFrameLineWidth(2)    # assi piu' spessi
ROOT.gStyle.SetLabelFont(42, "xyz")
ROOT.gStyle.SetTitleFont(42, "xyz")
ROOT.gStyle.SetLabelSize(SIZE_LABEL, "xyz")
ROOT.gStyle.SetTitleSize(SIZE_TITLE, "xyz")
ROOT.gStyle.SetTitleOffset(1.4, "xy")

# --- Sample colours ---
COLORS = {
    "00":    ROOT.TColor.GetColor("#0072B2"),   # blu     -> longitudinale (LL)
    "TT":    ROOT.TColor.GetColor("#C0392B"),   # rosso   -> trasversa (TT)
    "mixed": ROOT.TColor.GetColor("#7B3294"),   # viola   -> mista (LT)
    "bkg":   ROOT.TColor.GetColor("#666666"),   # grigio  -> fondo
    "sig":   ROOT.TColor.GetColor("#000000"),   # nero    -> segnale totale
}

# --- Colori per fit e scan di likelihood ---
COLORS_FIT = {
    "profilato":        ROOT.TColor.GetColor("#2171B5"),   # altri parametri profilati
    "profilato_chiaro": ROOT.TColor.GetColor("#9ECAE1"),
    "fisso":            ROOT.TColor.GetColor("#000000"),   # altri parametri fissi a 1
    "fisso_chiaro":     ROOT.TColor.GetColor("#808080"),
    "soglie":           ROOT.TColor.GetColor("#808080"),   # linee a 1 e 2 sigma
}

# --- Etichette ---
Y_TITLE = "Events / bin"
LABELS = {
    "00": "Longitudinal (00)",
    "TT": "Transverse (TT)",
    "mixed": "Mixed (0T + T0)",
    "bkg": "Background",
    "sig": "Signal",
}


# --- Funzioni grafiche ---
def make_canvas(name, title="", width=900, height=650):
    """Create a canvas with the standard margins used for the analysis plots."""
    c = ROOT.TCanvas(name, title, width, height)
    c.SetLeftMargin(0.17)
    c.SetBottomMargin(0.18)
    c.SetTopMargin(0.12)
    c.SetRightMargin(0.05)
    return c


# Keep TLatex objects alive while the canvases are displayed.
_FCC_LABEL_OBJECTS = []

def draw_fcc_label():
    """Draw the FCC-ee Simulation label, collision energy, and integrated luminosity."""
    latex = ROOT.TLatex()
    latex.SetNDC()
    latex.SetTextFont(42)
    latex.SetTextAlign(11)                  # allineato a sinistra
    latex.SetTextSize(SIZE_FCC)
    # Use one string with two fonts: #font[61] is bold and #font[52] is italic.
    t1 = latex.DrawLatex(0.24, 0.92, "#font[61]{FCC-ee} #font[52]{#scale[0.85]{Simulation}}")
    latex.SetTextAlign(31)                  # allineato a destra, sul bordo del grafico
    latex.SetTextSize(SIZE_FCC * 0.78)
    t2 = latex.DrawLatex(0.95, 0.92, f"#sqrt{{s}} = {COLLISION_ENERGY_GEV:g} GeV, L = {LUMI_PB / 1e6:g} ab^{{-1}}")
    _FCC_LABEL_OBJECTS.extend([latex, t1, t2])


def style_hist(hist, color, line_style=1, fill=True):
    """Set the histogram line colour and width, with an optional transparent fill."""
    hist.SetLineColor(color)
    hist.SetLineWidth(LINE_WIDTH)
    hist.SetLineStyle(line_style)
    if fill:
        hist.SetFillColorAlpha(color, 0.15)


def make_legend(x1, y1, x2, y2, header=None):
    """Create a legend without a border or background, with an optional header."""
    leg = ROOT.TLegend(x1, y1, x2, y2)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextFont(42)
    leg.SetTextSize(SIZE_LEGEND)
    leg.SetMargin(0.12)                     # meno spazio per il simbolo, piu' per il testo
    if header:
        leg.SetHeader(header)
    return leg


In [ ]:
# 1b. Helper functions

LABELS["bkg"] = (
    "Background (e^{+}e^{-} #rightarrow e^{+}#mu^{-}#nu#bar{#nu})"
    if SOLO_E_MU
    else "Background (e^{+}e^{-} #rightarrow l^{+}l^{-}#nu#bar{#nu})"
)


def leptoni_pdg(event):
    """Return the PDG IDs of charged electrons and muons with pT > 0.1 GeV."""
    pdg = []
    for p in range(event.nLHEPart):
        if abs(event.LHEPart_pdgId[p]) in (11, 13) and event.LHEPart_pt[p] > 0.1:
            pdg.append(int(event.LHEPart_pdgId[p]))
    return sorted(pdg)


def stato_finale_e_mu(event):
    """Return True if the charged final state is e+ mu-."""
    return leptoni_pdg(event) == [-11, 13]


def get_leptons(event):
    """
    Estrae i leptoni carichi e/mu con pT > 0.1 GeV.
    Restituisce una lista di coppie (PDG ID, TLorentzVector).
    """
    leptons = []

    for p in range(event.nLHEPart):
        pdg = int(event.LHEPart_pdgId[p])

        if abs(pdg) in (11, 13) and event.LHEPart_pt[p] > 0.1:
            v = ROOT.TLorentzVector()
            v.SetPtEtaPhiM(
                event.LHEPart_pt[p],
                event.LHEPart_eta[p],
                event.LHEPart_phi[p],
                event.LHEPart_mass[p]
            )
            leptons.append((pdg, v))

    return leptons


def get_lhe_particle_vector(event, index):
    """Build a four-vector for one particle in the LHE record."""
    v = ROOT.TLorentzVector()
    v.SetPtEtaPhiM(
        event.LHEPart_pt[index],
        event.LHEPart_eta[index],
        event.LHEPart_phi[index],
        event.LHEPart_mass[index]
    )
    return v


def calcola_cos_theta_star(lepton, sistema_riferimento):
    """
    Calcola cos(theta*):
    1. boost del leptone nel sistema di riposo del sistema di riferimento;
    2. angolo tra il leptone boostato e la direzione di moto
       del sistema di riferimento nel laboratorio.
    """
    asse = sistema_riferimento.Vect()

    if asse.Mag() <= 0:
        return None

    beta = sistema_riferimento.BoostVector()

    if beta.Mag2() >= 1.0:
        return None

    lepton_rest = ROOT.TLorentzVector(lepton)
    lepton_rest.Boost(-beta)

    p_lepton = lepton_rest.Vect()

    denominatore = p_lepton.Mag() * asse.Mag()

    if denominatore <= 0:
        return None

    cos_star = float(p_lepton.Dot(asse) / denominatore)

    if not math.isfinite(cos_star):
        return None

    return max(-1.0, min(1.0, cos_star))


def calcola_angoli_tmva(event, leptons, v_ll, is_signal):
    """
    Restituisce i due contributi angolari per evento.

    Segnale:
      - leptone positivo nel sistema di riposo del W+;
      - leptone negativo nel sistema di riposo del W-.

    Background:
      - entrambi i leptoni nel sistema di riposo dileptonico.

    La procedura riproduce quella usata per gli istogrammi angolari
    standard della likelihood.
    """

    if is_signal:

        # Identify the W+ and W- bosons in the LHE record.
        w_plus_indices = [
            p for p in range(event.nLHEPart)
            if event.LHEPart_pdgId[p] == 24
        ]

        w_minus_indices = [
            p for p in range(event.nLHEPart)
            if event.LHEPart_pdgId[p] == -24
        ]

        if len(w_plus_indices) != 1 or len(w_minus_indices) != 1:
            return None

        # Con la convenzione PDG:
        # pdg < 0 denotes a positively charged lepton;
        # pdg > 0 denotes a negatively charged lepton.
        leptoni_positivi = [
            v for pdg, v in leptons if pdg < 0
        ]

        leptoni_negativi = [
            v for pdg, v in leptons if pdg > 0
        ]

        if len(leptoni_positivi) != 1 or len(leptoni_negativi) != 1:
            return None

        w_plus = get_lhe_particle_vector(event, w_plus_indices[0])
        w_minus = get_lhe_particle_vector(event, w_minus_indices[0])

        cos_plus = calcola_cos_theta_star(
            leptoni_positivi[0],
            w_plus
        )

        cos_minus = calcola_cos_theta_star(
            leptoni_negativi[0],
            w_minus
        )

        if cos_plus is None or cos_minus is None:
            return None

        return [cos_plus, cos_minus]

    else:

        # Background: boost both leptons
        # to the dilepton rest frame.
        cos_values = []

        for pdg, lepton in leptons:
            cos_star = calcola_cos_theta_star(lepton, v_ll)

            if cos_star is None:
                return None

            cos_values.append(cos_star)

        if len(cos_values) != 2:
            return None

        return cos_values


def open_event_tree(filename):
    """Open a source ROOT file and find TreeS, or a TTree containing nLHEPart."""
    f_in = ROOT.TFile.Open(resolve_input(filename), "READ")

    if not f_in or f_in.IsZombie():
        raise OSError(f"Impossibile aprire il file ROOT: {filename}")

    tree_in = f_in.Get(BDT_TREE_NAME)

    if not tree_in:
        for key in f_in.GetListOfKeys():
            obj = f_in.Get(key.GetName())

            if obj and obj.InheritsFrom("TTree") and obj.GetBranch("nLHEPart"):
                tree_in = obj
                break

    if not tree_in:
        f_in.Close()
        raise KeyError(f"Nessun TTree con branch nLHEPart trovato in {filename}")

    return f_in, tree_in


def crea_tree_per_tmva(input_files, output_filename, is_signal):
    """
    Crea un tree TMVA con:
      - lep_pt
      - mll
      - cos_theta_star
      - is_train

    Per ogni evento selezionato vengono scritte due righe:
      - una per il primo contributo angolare;
      - una per il secondo contributo angolare.

    Le due righe dello stesso evento hanno gli stessi lep_pt, mll
    e is_train, ma valori distinti di cos_theta_star.

    is_train=1: campione destinato al training/test interno TMVA.
    is_train=0: holdout esterno, riservato ai template dello score.

    La suddivisione viene assegnata a livello di evento, prima
    di scrivere le due righe.
    """

    if isinstance(input_files, str):
        input_files = [input_files]

    f_out = ROOT.TFile.Open(output_path(output_filename), "RECREATE")

    if not f_out or f_out.IsZombie():
        raise OSError(f"Impossibile creare il file ROOT: {output_filename}")

    tree_out = ROOT.TTree("T", "Tree per TMVA")

    lep_pt_f = array.array('f', [0.0])
    mll_f = array.array('f', [0.0])
    cos_star_f = array.array('f', [0.0])
    is_train_f = array.array('i', [0])

    tree_out.Branch("lep_pt", lep_pt_f, "lep_pt/F")
    tree_out.Branch("mll", mll_f, "mll/F")
    tree_out.Branch("cos_theta_star", cos_star_f, "cos_theta_star/F")
    tree_out.Branch("is_train", is_train_f, "is_train/I")

    try:
        for filename in input_files:

            f_in, tree_in = open_event_tree(filename)

            n_all = int(tree_in.GetEntries())

            if n_all <= 0:
                f_in.Close()
                raise ValueError(f"Il tree di input è vuoto: {filename}")

            seed_file = SPLIT_SEED + zlib.crc32(filename.encode("utf-8"))
            rng = random.Random(seed_file)

            n_holdout = 0
            n_selected = 0
            n_selected_train = 0
            n_selected_holdout = 0
            n_rows_written = 0

            try:
                for event in tree_in:

                    # Assign each event to a subset only once.
                    train_flag = 1 if rng.random() < TRAIN_FRACTION else 0

                    if train_flag == 0:
                        n_holdout += 1

                    # Apply the e+ mu- restriction to the background only,
                    # and only when explicitly requested.
                    if (not is_signal) and SOLO_E_MU:
                        if not stato_finale_e_mu(event):
                            continue

                    leptons = get_leptons(event)

                    # Require exactly two charged electron/muon leptons.
                    if len(leptons) != 2:
                        continue

                    lepton_vectors = [v for pdg, v in leptons]

                    # Leading-lepton transverse momentum.
                    lep_pt = max(v.Pt() for v in lepton_vectors)

                    # Dilepton system and invariant mass.
                    v_ll = lepton_vectors[0] + lepton_vectors[1]
                    mll = float(v_ll.M())

                    if not math.isfinite(mll) or mll < 0:
                        continue

                    # Calculate the two angular contributions.
                    cos_values = calcola_angoli_tmva(
                        event,
                        leptons,
                        v_ll,
                        is_signal
                    )

                    if cos_values is None or len(cos_values) != 2:
                        continue

                    # Both angular contributions must be valid.
                    if not all(math.isfinite(c) for c in cos_values):
                        continue

                    lep_pt_f[0] = lep_pt
                    mll_f[0] = mll
                    is_train_f[0] = train_flag

                    # Store two consecutive rows for each event.
                    # This allows the subsequent block split
                    # to keep both rows from the same event together.
                    for cos_star in cos_values:
                        cos_star_f[0] = cos_star
                        tree_out.Fill()
                        n_rows_written += 1

                    n_selected += 1

                    if train_flag == 1:
                        n_selected_train += 1
                    else:
                        n_selected_holdout += 1

            finally:
                f_in.Close()

            holdout_fraction_by_file[filename] = n_holdout / n_all

            print(
                f"\n{filename}: {n_all} eventi sorgente, "
                f"eventi selezionati={n_selected} "
                f"(train={n_selected_train}, "
                f"holdout={n_selected_holdout}), "
                f"righe angolari scritte={n_rows_written}, "
                f"frazione holdout pre-tagli="
                f"{holdout_fraction_by_file[filename]:.4f}"
            )

    finally:
        f_out.cd()
        tree_out.Write()
        f_out.Close()

    print(f"[OK] Creato {output_filename}")


def n_eventi_generati(filename):
    """Return the generated-event count from the Events tree in the source file."""
    f = ROOT.TFile.Open(resolve_input(filename), "READ")

    if not f or f.IsZombie():
        raise OSError(f"Impossibile aprire {filename}")

    t = f.Get(EVENTS_TREE_NAME)

    if not t:
        f.Close()
        raise KeyError(f"Tree {EVENTS_TREE_NAME!r} non trovato in {filename}")

    n_gen = int(t.GetEntries())
    f.Close()

    if n_gen <= 0:
        raise ValueError(f"N_gen non positivo per {filename}")

    return n_gen


print("[OK] Setup e funzioni di supporto caricati.")

## 2. Prepare the event samples

The signal training sample combines the longitudinal (00), transverse (TT), and mixed (0T and T0) components. The background sample is prepared separately. Additional component-specific trees are retained for the likelihood templates.

The event split is assigned before the physics selection. Events marked `is_train = 1` (70%) are passed to TMVA for its internal training/test split. Events marked `is_train = 0` (30%) are reserved exclusively for the independent BDT-score templates used in the likelihood fit.


In [ ]:
# 2. Create the ROOT trees

# Clear the mapping if this cell is rerun in the same kernel.
holdout_fraction_by_file.clear()

file_segnale = [
    LEPTONIC_SAMPLE_FILES["decay_00"],
    LEPTONIC_SAMPLE_FILES["decay_0t"],
    LEPTONIC_SAMPLE_FILES["decay_t0"],
    LEPTONIC_SAMPLE_FILES["decay_tt"],
]

# Combined signal sample.
crea_tree_per_tmva(
    file_segnale,
    "tmva_signal.root",
    is_signal=True
)

# Background sample: retain all final states
# with exactly two selected charged e/mu leptons.
crea_tree_per_tmva(
    LEPTONIC_SAMPLE_FILES["background"],
    "tmva_background.root",
    is_signal=False
)

# Separate samples for the signal components.
crea_tree_per_tmva(
    LEPTONIC_SAMPLE_FILES["decay_00"],
    "tmva_signal_00.root",
    is_signal=True
)

crea_tree_per_tmva(
    LEPTONIC_SAMPLE_FILES["decay_tt"],
    "tmva_signal_TT.root",
    is_signal=True
)

crea_tree_per_tmva(
    LEPTONIC_SAMPLE_FILES["decay_0t"],
    "tmva_signal_0T.root",
    is_signal=True
)

crea_tree_per_tmva(
    LEPTONIC_SAMPLE_FILES["decay_t0"],
    "tmva_signal_T0.root",
    is_signal=True
)

print("\n[OK] Sample preparation completed.")

## 3. Train the binary BDT

TMVA trains a signal-versus-background classifier using `lep_pt`, `mll`, and `cos_theta_star`. Only events assigned to the 70% training sample enter TMVA. The independent 30% holdout sample remains excluded from the classifier training and is evaluated later when building the likelihood templates.


In [ ]:
# 3. Train the signal/background BDT

signal_filenames = {
    "LL": "tmva_signal_00.root",
    "TT": "tmva_signal_TT.root",
    "0T": "tmva_signal_0T.root",
    "T0": "tmva_signal_T0.root",
}

signal_files_open = {}
signal_trees = {}

for name, filename in signal_filenames.items():

    f = ROOT.TFile.Open(output_path(filename), "READ")

    if not f or f.IsZombie():
        raise OSError(f"Impossibile aprire {filename}")

    tree = f.Get("T")

    if not tree:
        f.Close()
        raise KeyError(f"Tree T non trovato in {filename}")

    n_train = int(tree.GetEntries("is_train == 1"))

    if n_train <= 0:
        f.Close()
        raise RuntimeError(f"Nessuna riga di training in {filename}")

    if n_train % 2 != 0:
        f.Close()
        raise RuntimeError(
            f"Numero dispari di righe di training in {filename}: "
            "le righe devono essere organizzate a coppie."
        )

    signal_files_open[name] = f
    signal_trees[name] = tree


f_bkg = ROOT.TFile.Open(output_path("tmva_background.root"), "READ")

if not f_bkg or f_bkg.IsZombie():
    raise OSError("Impossibile aprire tmva_background.root")

bkg_tree = f_bkg.Get("T")

if not bkg_tree:
    f_bkg.Close()
    raise KeyError("Tree T non trovato nel fondo")

n_bkg_train = int(bkg_tree.GetEntries("is_train == 1"))

if n_bkg_train <= 0:
    raise RuntimeError("Nessuna riga di training nel fondo")

if n_bkg_train % 2 != 0:
    raise RuntimeError(
        "Numero dispari di righe di training nel fondo: "
        "le righe devono essere organizzate a coppie."
    )


# Total signal weight for each component:
# LL=1, TT=1, Mixed=1 (0T=0.5, T0=0.5).
n_LL = signal_trees["LL"].GetEntries("is_train == 1")
n_TT = signal_trees["TT"].GetEntries("is_train == 1")
n_0T = signal_trees["0T"].GetEntries("is_train == 1")
n_T0 = signal_trees["T0"].GetEntries("is_train == 1")

signal_tree_weights = {
    "LL": 1.0 / n_LL,
    "TT": 1.0 / n_TT,
    "0T": 0.5 / n_0T,
    "T0": 0.5 / n_T0,
}

print("Pesi del training binario:")

for name, weight in signal_tree_weights.items():
    n = signal_trees[name].GetEntries("is_train == 1")
    print(
        f"  {name}: N righe={n}, "
        f"peso/riga={weight:.6e}, "
        f"peso totale={n * weight:.3f}"
    )


# Total number of rows available for each class.
# Each event contributes two rows, so the counts
# must be even.
n_signal_entries = sum(
    tree.GetEntries("is_train == 1")
    for tree in signal_trees.values()
)

n_background_entries = n_bkg_train


def split_numero_pari(n_entries, nome):
    """
    Divide le righe in training e test mantenendo pari
    entrambi i conteggi.

    Poiché ogni evento è rappresentato da due righe consecutive,
    la divisione a blocchi non separa i due contributi dello stesso evento.
    """
    if n_entries % 2 != 0:
        raise RuntimeError(
            f"Numero dispari di righe per {nome}: {n_entries}"
        )

    n_train = 2 * (n_entries // 4)
    n_test = n_entries - n_train

    if n_train < 2 or n_test < 2:
        raise RuntimeError(
            f"Campione {nome} insufficiente per una divisione "
            f"training/test a coppie: N={n_entries}"
        )

    return n_train, n_test


nTrain_Signal, nTest_Signal = split_numero_pari(
    n_signal_entries,
    "segnale"
)

nTrain_Background, nTest_Background = split_numero_pari(
    n_background_entries,
    "background"
)

print("\nSuddivisione interna TMVA:")

print(
    f"  Segnale: training={nTrain_Signal}, "
    f"test={nTest_Signal}"
)

print(
    f"  Background: training={nTrain_Background}, "
    f"test={nTest_Background}"
)


output_file = ROOT.TFile.Open(
    output_path("TMVA_Binario_Output.root"),
    "RECREATE"
)

if not output_file or output_file.IsZombie():
    raise OSError("Impossibile creare TMVA_Binario_Output.root")


factory = ROOT.TMVA.Factory(
    "AnalisiTesi",
    output_file,
    "!V:!Silent:Color:DrawProgressBar:AnalysisType=Classification"
)

dataloader = ROOT.TMVA.DataLoader("dataset")

# Keep the same three classifier input variables.
dataloader.AddVariable("lep_pt", "F")
dataloader.AddVariable("mll", "F")
dataloader.AddVariable("cos_theta_star", "F")


for name in ["LL", "TT", "0T", "T0"]:
    dataloader.AddSignalTree(
        signal_trees[name],
        signal_tree_weights[name]
    )

dataloader.AddBackgroundTree(bkg_tree, 1.0)


cut_training = ROOT.TCut("is_train == 1")

# SplitMode=Block:
# the first nTrain rows are used for training,
# and the following nTest rows are used for testing.
#
# The row counts are even and each event contributes
# two consecutive rows, keeping each pair in one subset.
dataloader.PrepareTrainingAndTestTree(
    cut_training,
    cut_training,
    (
        f"nTrain_Signal={nTrain_Signal}:"
        f"nTest_Signal={nTest_Signal}:"
        f"nTrain_Background={nTrain_Background}:"
        f"nTest_Background={nTest_Background}:"
        "SplitMode=Block:"
        "NormMode=EqualNumEvents:!V"
    )
)


factory.BookMethod(
    dataloader,
    ROOT.TMVA.Types.kBDT,
    "BDT",
    "!H:!V:NTrees=200:MinNodeSize=2.5%:MaxDepth=3:"
    "BoostType=AdaBoost:AdaBoostBeta=0.5:"
    "SeparationType=GiniIndex:nCuts=30"
)


factory.TrainAllMethods()
factory.TestAllMethods()
factory.EvaluateAllMethods()

output_file.Close()

for f in signal_files_open.values():
    f.Close()

f_bkg.Close()

print("\n[OK] Training BDT binario completato.")
print("TMVA weights: dataset/weights/AnalisiTesi_BDT.weights.xml")

In [ ]:
# 4. Binary BDT output (test sample): signal versus background, with uncertainties

def score_tmva(f_tmva, dataset, albero, class_id, nome):
    """Build a BDT-score histogram from the TMVA TestTree or TrainTree.

    class_id: 0 = segnale, 1 = fondo.
    Sumw2 e' attivo, quindi le incertezze statistiche sono corrette anche con eventi pesati.
    Si fa il rebin (stessi bin dei template della likelihood) e poi si normalizza ad area 1.
    """
    tree = f_tmva.Get(f"{dataset}/{albero}")
    if not tree:
        raise KeyError(f"{dataset}/{albero} non trovato nel file TMVA")
    h = ROOT.TH1F(nome, ";BDT score;Normalized events / bin", BINS_BDT, -1.0, 1.0)
    h.Sumw2()
    for evento in tree:
        if evento.classID == class_id:
            h.Fill(evento.BDT, evento.weight)
    h.Rebin(REBIN_BDT)
    if h.Integral() <= 0:
        raise RuntimeError(f"Istogramma vuoto: {nome}")
    h.Scale(1.0 / h.Integral())
    return h


f_tmva = ROOT.TFile.Open(output_path("TMVA_Binario_Output.root"), "READ")
if not f_tmva or f_tmva.IsZombie():
    raise OSError("Impossibile aprire TMVA_Binario_Output.root")

h_sig_test = score_tmva(f_tmva, "dataset", "TestTree", 0, "h_sig_test")
h_bkg_test = score_tmva(f_tmva, "dataset", "TestTree", 1, "h_bkg_test")
f_tmva.Close()

style_hist(h_sig_test, COLORS["sig"])
style_hist(h_bkg_test, COLORS["bkg"])

c_bdt = make_canvas("c_bdt_output", "BDT Output Distribution")
c_bdt.cd()

max_val = max(h_sig_test.GetMaximum(), h_bkg_test.GetMaximum())
h_sig_test.GetYaxis().SetRangeUser(0.0, max_val * 1.5)
h_sig_test.Draw("HIST E")
h_bkg_test.Draw("HIST E SAME")

leg_bdt = make_legend(0.40, 0.74, 0.95, 0.86)
leg_bdt.AddEntry(h_sig_test, LABELS["sig"] + " (test sample)", "lf")
leg_bdt.AddEntry(h_bkg_test, "Background (test sample)", "lf")
leg_bdt.Draw()

draw_fcc_label()
c_bdt.RedrawAxis()
c_bdt.Modified()
c_bdt.Update()
c_bdt.Draw()

ROOT_KEEPALIVE.extend([c_bdt, leg_bdt, h_sig_test, h_bkg_test])

print("[OK] Distribuzione BDT del test: Signal vs Background.")


In [ ]:
# 5. Build BDT templates and perform the likelihood fit

reader = ROOT.TMVA.Reader("!Color:!Silent")

v_pt = array.array('f', [0.0])
v_mll = array.array('f', [0.0])
v_cos = array.array('f', [0.0])

reader.AddVariable("lep_pt", v_pt)
reader.AddVariable("mll", v_mll)
reader.AddVariable("cos_theta_star", v_cos)

reader.BookMVA(
    "BDT",
    "dataset/weights/AnalisiTesi_BDT.weights.xml"
)


def get_bdt_histogram(contributions, hist_name):
    """
    Costruisce un template pesato usando gli eventi del holdout esterno.

    contributions: lista di tuple
        (file_preparato, file_originale, sigma_pb).

    Ogni evento contribuisce con due righe angolari, quindi il template
    viene riempito due volte per ciascun evento selezionato.
    """

    hist = ROOT.TH1F(
        hist_name,
        f";BDT score;{Y_TITLE}",
        BINS_BDT,
        -1.0,
        1.0
    )

    hist.Sumw2()
    hist.SetDirectory(0)

    for i_contrib, (prepared_file, original_file, sigma) in enumerate(contributions):

        frac = holdout_fraction_by_file.get(original_file)

        if frac is None or frac <= 0:
            raise ValueError(
                f"Frazione holdout non disponibile per {original_file}"
            )

        n_gen = n_eventi_generati(original_file)
        weight = sigma * LUMI_PB / (n_gen * frac)

        h_part = ROOT.TH1F(
            f"{hist_name}_part{i_contrib}",
            f";BDT score;{Y_TITLE}",
            BINS_BDT,
            -1.0,
            1.0
        )

        h_part.Sumw2()
        h_part.SetDirectory(0)

        f_in = ROOT.TFile.Open(output_path(prepared_file), "READ")

        if not f_in or f_in.IsZombie():
            raise OSError(f"Impossibile aprire {prepared_file}")

        try:
            tree = f_in.Get("T")

            if not tree:
                raise KeyError(f"Tree T non trovato in {prepared_file}")

            is_train = array.array('i', [0])

            for branch, var in [
                ("lep_pt", v_pt),
                ("mll", v_mll),
                ("cos_theta_star", v_cos),
                ("is_train", is_train),
            ]:
                status = tree.SetBranchAddress(branch, var)

                if status < 0:
                    raise RuntimeError(
                        f"Branch {branch} non disponibile in {prepared_file}"
                    )

            n_righe_holdout = 0

            for i in range(tree.GetEntries()):

                tree.GetEntry(i)

                if is_train[0] != 0:
                    continue

                score = float(reader.EvaluateMVA("BDT"))

                h_part.Fill(score)
                n_righe_holdout += 1

        finally:
            f_in.Close()

        if n_righe_holdout == 0:
            raise RuntimeError(
                f"Nessuna riga holdout selezionata in {original_file}"
            )

        if n_righe_holdout % 2 != 0:
            raise RuntimeError(
                f"Numero dispari di righe holdout in {original_file}: "
                "manca uno dei due contributi angolari."
            )

        h_part.Scale(weight)
        hist.Add(h_part)

        print(
            f"{original_file}: N_gen={n_gen}, "
            f"frazione holdout={frac:.4f}, "
            f"righe angolari holdout={n_righe_holdout}, "
            f"eventi holdout={n_righe_holdout // 2}, "
            f"peso per riga={weight:.6g}"
        )

    if hist.Integral() <= 0:
        raise RuntimeError(f"Template vuoto: {hist_name}")

    return hist


# Templates for the three signal components and the background.
h_LL = get_bdt_histogram(
    [
        ("tmva_signal_00.root", LEPTONIC_SAMPLE_FILES["decay_00"], SIGMA_LL)
    ],
    "h_bdt_LL"
)

h_TT = get_bdt_histogram(
    [
        ("tmva_signal_TT.root", LEPTONIC_SAMPLE_FILES["decay_tt"], SIGMA_TT)
    ],
    "h_bdt_TT"
)

h_LT = get_bdt_histogram(
    [
        ("tmva_signal_0T.root", LEPTONIC_SAMPLE_FILES["decay_0t"], SIGMA_0T),
        ("tmva_signal_T0.root", LEPTONIC_SAMPLE_FILES["decay_t0"], SIGMA_T0),
    ],
    "h_bdt_LT"
)

h_bkg = get_bdt_histogram(
    [
        ("tmva_background.root", LEPTONIC_SAMPLE_FILES["background"], SIGMA_BKG)
    ],
    "h_bdt_bkg"
)


# Rebin by combining REBIN_BDT adjacent bins.
for h_template in (h_LL, h_TT, h_LT, h_bkg):
    h_template.Rebin(REBIN_BDT)


N_BINS = h_bkg.GetNbinsX()

template_bkg = np.array(
    [h_bkg.GetBinContent(i) for i in range(1, N_BINS + 1)],
    dtype=float
)

template_LL = np.array(
    [h_LL.GetBinContent(i) for i in range(1, N_BINS + 1)],
    dtype=float
)

template_TT = np.array(
    [h_TT.GetBinContent(i) for i in range(1, N_BINS + 1)],
    dtype=float
)

template_LT = np.array(
    [h_LT.GetBinContent(i) for i in range(1, N_BINS + 1)],
    dtype=float
)


# Asimov dataset: sum of the nominal templates.
data = template_bkg + template_LL + template_TT + template_LT


def negative_log_likelihood(mu_00, mu_tt, mu_mix):

    expected = (
        template_bkg
        + mu_00 * template_LL
        + mu_tt * template_TT
        + mu_mix * template_LT
    )

    mask = data > 0

    if np.any(expected[mask] <= 0):
        return 1e100

    term = np.zeros_like(data)
    term[mask] = data[mask] * np.log(data[mask] / expected[mask])

    return float(2.0 * np.sum(expected - data + term))


fit_bdt = Minuit(
    negative_log_likelihood,
    mu_00=1.0,
    mu_tt=1.0,
    mu_mix=1.0
)

fit_bdt.errordef = 1.0

for par in ("mu_00", "mu_tt", "mu_mix"):
    fit_bdt.limits[par] = (0.0, 5.0)

fit_bdt.migrad()
fit_bdt.hesse()


print("\n========== FIT LIKELIHOOD — BDT BINARIO ==========")

print(
    f"μ_LL = {fit_bdt.values['mu_00']:.5f} "
    f"± {fit_bdt.errors['mu_00']:.5f}"
)

print(
    f"μ_TT = {fit_bdt.values['mu_tt']:.5f} "
    f"± {fit_bdt.errors['mu_tt']:.5f}"
)

print(
    f"μ_LT = {fit_bdt.values['mu_mix']:.5f} "
    f"± {fit_bdt.errors['mu_mix']:.5f}"
)

print(f"Fit valid: {'yes' if fit_bdt.valid else 'no'}")

print(
    f"Covarianza accurata: "
    f"{'sì' if getattr(fit_bdt.fmin, 'has_accurate_covar', False) else 'no'}"
)

print(f"EDM: {fit_bdt.fmin.edm:.3g}")


if fit_bdt.covariance is not None:

    corr = fit_bdt.covariance.correlation()

    print("Correlations:")
    print(f"  ρ(μ_LL, μ_TT) = {corr['mu_00', 'mu_tt']:+.4f}")
    print(f"  ρ(μ_LL, μ_LT) = {corr['mu_00', 'mu_mix']:+.4f}")
    print(f"  ρ(μ_TT, μ_LT) = {corr['mu_tt', 'mu_mix']:+.4f}")

else:
    print("[WARNING] Matrice di covarianza non disponibile.")


ROOT_KEEPALIVE.extend([h_LL, h_TT, h_LT, h_bkg])

In [ ]:
# Save all canvases as vector PDF and PNG files
# PDF preserves vector quality when figures are resized.
# Run this cell after creating the canvases to export all figures.
import os
for canvas in ROOT.gROOT.GetListOfCanvases():
    canvas.SaveAs(output_path(f"plots/{canvas.GetName()}.pdf"))
    canvas.SaveAs(output_path(f"plots/{canvas.GetName()}.png"))
